# 39 · ColBERT / SPLADE / 长上下文

> 稠密向量丢失词级信号，稀疏向量不懂语义。**ColBERT**（多向量）与 **SPLADE**（词级+学习）在补足这块。本课顺带聊**长上下文时代**对 RAG 的冲击。

**本文件覆盖知识点**：ColBERT / MaxSim / Late Interaction / SPLADE / 稀疏与稠密结合 / Long Context / Lost in the Middle

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. ColBERT：Late Interaction + MaxSim

```text
Bi-Encoder:  query→1个向量 × doc→1个向量 = 点积(粗，丢词)   
ColBERT:     query→N个词向量  doc→M个词向量
评分 = 每个 query 词向量 去 doc 里找最相似的词向量(MaxSim) 再求和
→ 保留了“哪个词对上哪个词”的精确度，又享受了向量检索的速度
```

- **Late Interaction**：交互放在最后打分阶段（不做全量交叉，仍可走 ANN 预筛）；
- **MaxSim**：query 每个 token 与 doc 全部 token 的最大相似度之和。

### ColBERT 的工程配合
- 先 ANN 召回粗候选 → 只对候选做 ColBERT 精排 → 精度/成本折中；
- 需存储“文档的多向量”，索引体积比普通稠密大，可用残差压缩。

In [ ]:
# 用手工向量演示 MaxSim（理解 Late Interaction 的评分直觉）
import numpy as np

# 假设 embed_token 把每个词映射到向量；此处用随机但固定的矩阵模拟
rng = np.random.default_rng(0)
vocab = {'苹果', '价格', '上涨', '天气', '很好', '晴天'}
w2v = {w: rng.normal(size=8) for w in vocab}
w2v = {w: v/np.linalg.norm(v) for w, v in w2v.items()}  # 归一化

def colbert_score(q_tokens, d_tokens):
    sims = np.array([[w2v[q] @ w2v[d] for d in d_tokens] for q in q_tokens])
    maxsim = sims.max(axis=1)          # 每个 query 词 → doc 中最佳匹配
    return maxsim.sum(), maxsim.round(2)

q = ['苹果', '价格']
s1, m1 = colbert_score(q, ['苹果', '价格', '上涨'])
s2, m2 = colbert_score(q, ['天气', '很好', '晴天'])
print('相似文档打分:', s1.round(2), m1)
print('无关文档打分:', s2.round(2), m2)
print('=> MaxSim 让每个查询词都“各自找得到同伴”，无关文档匹配不到高分。')

## 2. SPLADE：学出来的稀疏向量

SPLADE 用模型把文本变成**稀疏的 term 加权向量**（含同义扩展词），兼得：
- BM25 的可解释/词级精确；
- 稠密向量的语义扩展（“小轿车”也能激活“汽车”）。

```text
输入: 汽车修理
输出稀疏向量: 汽车:1.2  轿车:0.8  维修:1.0  保养:0.6 …（其它全 0）
→ 可直接放进倒排索引，与 BM25 系出同门
```

### 怎么选
| 模型 | 适用 | 代价 |
|------|------|------|
| 稠密向量 | 语义相似为主 | 低（1 向量/文档） |
| **ColBERT** | 高精度精排/复杂匹配 | 多向量、存储高 |
| **SPLADE** | 需词级精确+语义、可解释 | 推理成本较高 |

In [ ]:
# 知识点·真调说明：检索模型选型 —— 三个业务场景让模型对号入座讲权衡
_llm_live(
    prompt='为下面三个检索场景各选一个主方案（稠密向量 / ColBERT / SPLADE / 混合），并用一句话说理由：\n'
           '① 法律合同：必须精确命中条款编号与原文措辞，查不到即错；\n'
           '② 电商“找同款”：买家描述五花八门，语义相近即可；\n'
           '③ 故障工单：问题口语化，又要兼顾“机型/报错码”等精确词。',
    system='你是检索系统架构师。请按“①选型：理由”逐条回答，直接给结论，不展开教材式背景。',
    fallback='未配置 Key 的固定样例：\n'
             '① 法律合同：SPLADE，或 SPLADE/BM25 保词级命中 + 稠密语义兜底的混合——条款编号只能靠词级命中。\n'
             '② 电商同款：稠密向量主 + ColBERT 精排——语义为主，多向量只在精排阶段补精确匹配。\n'
             '③ 故障工单：混合检索（稀疏命中型号/报错码 + 稠密接住口语化描述），存储与延迟允许再叠加 ColBERT 精排。',
    temperature=0.2,
)
print('→ 没有“最好的检索模型”：数据形态决定该保“词级”还是“语义”；工程上通常是几者组合而非单选。')

## 3. Long Context（长上下文）对 RAG 的影响

模型窗口越来越长（qwen-long 等达百万 token），带来新选择：

- 直接把文档塞进窗口的 **Long-Context 模式**：省检索、防漏信息；但贵、慢；
- **Lost in the Middle**：长上下文里模型对“中间位置”的信息记得最差——
  - 关键片段放**开头/结尾**；
  - 或仍用 RAG 把长文档压缩成最相关段落，再进窗口。

> 实践倾向：长上下文与 RAG **互补**——RAG 选段 + 长窗口兜底“还要更多上下文”时。



In [ ]:
# 知识点·真调说明：Lost-in-the-Middle —— 唯一答案放“开头 vs 正中间”，让长上下文找同一件事
_a = ['华鑫', '森屿', '远帆', '明澈', '嘉禾', '蓝湾', '北辰', '清晏', '卓然', '隽永', '瀚海', '青梧',
      '澄川', '曜石', '栖云', '赤松', '聆风', '沐光', '逸云', '观澜', '擎苍', '归鸿', '凝霜', '拂晓',
      '叠翠', '惊鸿', '鸣沙', '临渊', '竹影', '松风']
_suf = ['科技', '数据', '物联', '智造', '云服', '软件']
_svc = ['质检云', '设计库', '货运SaaS', '病历检索', '能源台账', '楼宇物联', '题库系统', '基因检索',
        '设备云', '文创库', '云盘', '农情监测', '水务台账', '安全监测', '点餐系统', '坯布管理',
        '行程助手', '健康档案', '车服百科', '保单库', '产线MES', '进销存', '代码检索', '咨询知识库',
        '苗木台账', '媒资库', '运单系统', '安防平台', '电子书库', '生产看板']

def _t(i):
    m = 11 + i // 28          # 分布在 2026-11 ~ 2027-05 之间
    y = 2026
    if m > 12:
        m -= 12
        y = 2027
    d = i % 28 + 1
    if (y, m, d) == (2027, 3, 18):   # 别把唯一目标日期 2027-03-18 也生成进干扰项
        d = 19
    return '%d-%02d-%02d' % (y, m, d)

_other = [(_a[i % 30] + _suf[i // 30], _svc[i % 30], _t(i)) for i in range(180)]
_target = ('启明智造', '设备巡检', '2027-03-18')

def _records(target_first):
    if target_first:
        items = [_target] + _other
    else:
        k = len(_other) // 2
        items = _other[:k] + [_target] + _other[k:]
    return '\n'.join('客户「%s」签约了「%s」，合同续约时间：%s。' % t for t in items)

_q = '请只输出一个日期（YYYY-MM-DD）：客户「启明智造」的合同续约时间是？'
print('① 唯一答案放最开头（共 181 条相似记录，约 5k token）')
head = _llm_live(
    prompt=_records(True) + '\n\n' + _q,
    system='你是信息提取助手，只依据给定资料回答，不要输出日期以外的任何文字。',
    fallback='未配置 Key 的固定样例：\n2027-03-18（答案在开头，轻松命中）',
    temperature=0.1,
)
print()
print('② 唯一答案埋在 181 条相似记录的正中间')
mid = _llm_live(
    prompt=_records(False) + '\n\n' + _q,
    system='你是信息提取助手，只依据给定资料回答，不要输出日期以外的任何文字。',
    fallback='未配置 Key 的固定样例：\n2027-01-16（长上下文里中间位置的唯一答案被干扰项淹没——'
             '同一句话放到开头就不会错，这就是 Lost-in-the-Middle）',
    temperature=0.1,
)
print()
def _hit(s):
    return s is not None and '2027-03-18' in s

if not _HAS_KEY:
    print('（无 Key 演示样例效果：开头版答对、中间版答错 → Lost-in-the-Middle）')
elif _hit(head) and _hit(mid):
    print('本次两处都答对：模型在更长上下文里依然把中间信息找了出来。'
          'Lost-in-the-Middle 是概率性现象：越长、噪声越多、越靠中间越容易丢，不是必然翻车。')
elif _hit(head) and not _hit(mid):
    print('→ 现场复现 Lost-in-the-Middle：同一答案放开头能答对，埋进正中间就被干扰带偏——位置确实影响命中。')
else:
    print('本次结果有波动（可重跑观察）。结论不变：上下文越长，越靠中间的信息越容易被模型忽略。')
print('→ 工程启示：窗口再长也不要“一股脑全塞”——先用 RAG 挑出最相关片段、把关键句放靠前位置，再进生成。')

## 小结

- **ColBERT** 用 MaxSim 晚交互保留词级精确度；
- **SPLADE** 用学习型稀疏向量兼得语义与词级；
- **长上下文**不是 RAG 的终点，注意 Lost-in-the-Middle，两者结合更优。